In [10]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. قراءة البيانات وتوحيد أسماء الأعمدة
df = pd.read_csv("ml.csv")
df.columns = df.columns.str.strip().str.lower()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 2. تنظيف قيم Unavailable وتحويل الأعمدة الرقمية
df = df.replace(to_replace=r"(?i)^unavailable$", value=np.nan, regex=True)

for col in df.columns:
  if col != "timestamp":
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "is_connected" in df.columns:
  df["is_connected"] = df["is_connected"].replace({
      True: 1,
      False: 0,
      "True": 1,
      "False": 0,
      "true": 1,
      "false": 0,
  })
  df["is_connected"] = pd.to_numeric(df["is_connected"], errors="coerce")

# 3. تحديد الأهداف والمدخلات
target_cols = ["rsrp", "rssi"]
for col in target_cols:
  if col not in df.columns:
    raise ValueError(f"Missing target column: {col}")

time_col = "rel_time" if "rel_time" in df.columns else "rel-time"
if time_col not in df.columns:
  raise ValueError("Missing time column ('rel_time' or 'rel-time')")

drop_cols = ["rsrp", "rssi", "asu", "ci", "tac", "timestamp"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
y = df[target_cols].copy()

# حذف الصفوف التي تفتقر لقيم الأهداف أو الزمن
valid_rows = y.notna().all(axis=1) & df[time_col].notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1, how="all")

# 4. الترتيب الزمني والتقسيم (80% تدريب / 20% اختبار)
X = X.sort_values(time_col)
y = y.loc[X.index]

split_index = int(len(X) * 0.8)
X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.loc[X_train.index]
y_test = y.loc[X_test.index]

# 5. تعويض القيم الناقصة باستخدام وسيط التدريب فقط
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("\nData preparation completed!")
print("Features used:", X_train.columns.tolist())
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

# 6. تعريف الموديلات
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
}

# 7. التدريب، عرض ملخص المقاييس، وحفظ الموديلات والتوقعات للعضو رقم 4
for model_name, model in models.items():
  print(f"\n==================== Training {model_name} ====================")
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  for idx, col in enumerate(target_cols):
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx]))
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    print(f"[{col.upper()}] MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

  joblib.dump(
      {
          "model": model,
          "features": X_train.columns.tolist(),
          "medians": train_medians,
      },
      f"{model_name.lower()}_rsrp_rssi.joblib",
  )

  results = pd.DataFrame(
      {
          "Actual_RSRP": y_test["rsrp"],
          "Predicted_RSRP": y_pred[:, 0],
          "Actual_RSSI": y_test["rssi"],
          "Predicted_RSSI": y_pred[:, 1],
      },
      index=y_test.index,
  )
  results.to_csv(f"{model_name.lower()}_predictions.csv", index=True)
  print(
      f"Saved: {model_name.lower()}_rsrp_rssi.joblib & "
      f"{model_name.lower()}_predictions.csv"
  )

print("\nAll models trained and saved successfully!")

Dataset shape: (56602, 15)
Columns: ['timestamp', 'rel_time', 'longitude', 'latitude', 'altitude', 'pci', 'ci', 'is_connected', 'tac', 'earfcn', 'rsrp', 'rsrq', 'rssi', 'asu', 'ta']

Data preparation completed!
Features used: ['rel_time', 'longitude', 'latitude', 'altitude', 'pci', 'is_connected', 'earfcn', 'rsrq', 'ta']
X_train: (45281, 9) | X_test: (11321, 9)
y_train: (45281, 2) | y_test: (11321, 2)

==================== Training RandomForest ====================
[RSRP] MAE: 7.8035 | RMSE: 10.0814 | R2: 0.2938
[RSSI] MAE: 7.3690 | RMSE: 9.7288 | R2: 0.3427
Saved: randomforest_rsrp_rssi.joblib & randomforest_predictions.csv

==================== Training ExtraTrees ====================
[RSRP] MAE: 4.6385 | RMSE: 5.8066 | R2: 0.7657
[RSSI] MAE: 4.5317 | RMSE: 5.8351 | R2: 0.7635
Saved: extratrees_rsrp_rssi.joblib & extratrees_predictions.csv

All models trained and saved successfully!


In [11]:
import time
from google.colab import files

files_to_download = [
    "randomforest_rsrp_rssi.joblib",
    "randomforest_predictions.csv",
    "extratrees_rsrp_rssi.joblib",
    "extratrees_predictions.csv",
]

for f in files_to_download:
  files.download(f)
  time.sleep(1)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>